Gets the predicted values for scLEMBAS models.

Loss is assessed with EMD for single-cell measurements (for comparison to single-cell baselines random and no adversarial removal)

For psuedobulk (to compare to RF, linear, and train mean baselines), loss is calculated after psuedobulking either by perturbation or condition (cell type x perturbation) using the mean of the sample-wise RMSE (as described in [Ahlmann-Eltze et al](https://doi.org/10.1038/s41592-025-02772-6) and the Pearson delta (used in [Ahlmann-Eltze et al](https://doi.org/10.1038/s41592-025-02772-6) and [Csendes et al](https://doi.org/10.1186/s12864-025-11600-2)). 

For prediction visualization, we project predictions into the PLS fit on the global actual data TF activity space from Notebook 04. 

In [3]:
import os
import collections
import itertools

import numpy as np
import pandas as pd

import copy

import sys
sys.path.insert(1, '../.')
from McCauley_utils import all_data

sys.path.insert(1, '../../.') 
from notebook_utils import get_split

sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
from scLEMBAS.predict import (
    get_prediction, 
    merge_novar_predictions, 
    merge_ctrl_with_pert, 
    project_pls_per_condition)

from scLEMBAS.metrics import distances 

/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install

In [4]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'McCauley'

n_cores = 30
os.environ["OMP_NUM_THREADS"] = str(1)
os.environ["MKL_NUM_THREADS"] = str(1)
os.environ["OPENBLAS_NUM_THREADS"] = str(1)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(1)
os.environ["NUMEXPR_NUM_THREADS"] = str(1)

In [5]:
(sn_ppis, tf_adata, adata, expr, source_label, target_label, weight_label, 
 stimulation_label, inhibition_label, cat_col, pert_col, ctrl_pert) = all_data

gt_map = {pert_col: 'perturbation', 'condition': 'condition'}

# Single-cell losses

Here, we get the fitted scLEMBAS model predictions. Next, we get the EMD for the scLEMBAS models (full model, random baseline, no adversarial training baseline). 

For the full model, it will also be compared to psuedobulk baselines (RF, linear, training mean), so we will also calculate the pseudo-bulk RMSE and Pearson delta metrics.

In [4]:
global_bias_removed = [['adj', 'global_bias'], 'global_bias', 'total_bias']

folds = range(5)
remove_types = [
    'none',
    ['adj', 'categorical_bias'],
    ['adj', 'global_bias'],
    'total_bias', 
    'adj',
    'categorical_bias',
    'global_bias'
]
mod_types = ['actual', 'random', 'noadv']

In [5]:
def prediction_pipeline(fold, mod_type, remove_type):
    split = get_split(fold = fold, author = author)

    test_conds = split['test_conds']
    train_barcodes = split['train_barcodes']
    test_barcodes = split['test_barcodes']

    # load trainer
    fn_trainer = os.path.join(data_path, 'processed', '{}_fold{}trainer_{}.pickle'.format(author, fold, mod_type))
    trainer = io.read_pickled_object(fn_trainer)
    mod = trainer.mod
    assert trainer.X_train.index.tolist() == train_barcodes, 'Train barcodes mismatch'
    assert trainer.X_test.index.tolist() == test_barcodes, 'Test barcodes mismatch'

    # get the control cells (from which counterfactual was predicted)
    ctrl_conds = sorted(set([tc.split('^')[0] + '^' + ctrl_pert for tc in test_conds]))
    ctrl_mask = tf_adata.obs.loc[train_barcodes, 'condition'].isin(ctrl_conds).values
    ctrl_cells = list(np.array(train_barcodes)[ctrl_mask])

    loss_res = collections.defaultdict(list)
    predictions_res = None if mod_type != 'actual' else {}

    tf_adata_predicted = get_prediction(
        mod = mod,
        train_cells = train_barcodes,
        test_cells = test_barcodes, 
        tf_adata = tf_adata,
        cat_col = cat_col,
        pert_col = pert_col,
        ctrl_pert = ctrl_pert, 
        counterfactual = 'perturbation', # counterfactual from tests
        cat_counterfactual_map = None,
        remove_type = remove_type,
        return_bias = False, 
        max_cells = int(5e3), 
        return_full = False, 
        stim_map = None, # special use case for Kang
    )

    # calculate the losses
    emd_loss = distances.get_EMD_loss(tf_adata, tf_adata_predicted)['Mean EMD Loss']
    loss_res['loss'].append(emd_loss)
    loss_res['loss_type'].append('EMD')
    loss_res['remove_type'].append(remove_type)
    loss_res['fold'].append(fold)
    loss_res['model_type'].append(mod_type)

    if mod_type == 'actual':
        if remove_type == 'none':
            for groupby_col in [pert_col, 'condition']:
                rmse_loss = distances.get_rmse(tf_adata, tf_adata_predicted, groupby_col = groupby_col)
                loss_res['loss'].append(rmse_loss)
                loss_res['loss_type'].append('RMSE_{}'.format(groupby_col))
                loss_res['remove_type'].append(remove_type)
                loss_res['fold'].append(fold)
                loss_res['model_type'].append(mod_type)

                pd_loss = distances.get_pearson_delta(tf_adata, tf_adata_predicted, groupby_col = groupby_col, 
                                                  groupby_type = gt_map[groupby_col], 
                                                  ctrl_pert = ctrl_pert)
                loss_res['loss'].append(pd_loss)
                loss_res['loss_type'].append('PearsonDelta_{}'.format(groupby_col))
                loss_res['remove_type'].append(remove_type)
                loss_res['fold'].append(fold)
                loss_res['model_type'].append(mod_type)


        # CTRL: get the control condition predictions (training data with no counterfactual) as a comparison
        tf_adata_ctrl_predicted = get_prediction(
            mod = mod,
            train_cells = ctrl_cells,
            test_cells = [], 
            tf_adata = tf_adata,
            cat_col = cat_col,
            pert_col = pert_col,
            ctrl_pert = ctrl_pert, 
            counterfactual = None,
            cat_counterfactual_map = None,
            remove_type = remove_type,
            return_bias = False, 
            max_cells = int(5e3), 
            return_full = False,
            stim_map = None, # special use case for Kang
        )

        # check lack of variance in non-generative predictions (no b_g) and merge
        if remove_type in global_bias_removed:
            tf_adata_predicted = merge_novar_predictions(tf_adata_predicted, remove_type, 
                                                         cat_col = cat_col, pert_col = pert_col, 
                                                         atol = 1e-4, rtol = 1e-8)
            # CTRL:
            tf_adata_ctrl_predicted = merge_novar_predictions(tf_adata_ctrl_predicted, remove_type, 
                                                              cat_col = cat_col, pert_col = pert_col, 
                                                              atol = 1e-4, rtol = 1e-8)

        tf_adata_merged = merge_ctrl_with_pert(
            tf_adata_actual = tf_adata, 
            tf_adata_pert_predicted = tf_adata_predicted, 
            tf_adata_ctrl_predicted = tf_adata_ctrl_predicted, 
            cat_col = cat_col, 
            pert_col = pert_col
        )

        # use the global PLS (not per-condition)
        pls_models = {tc: copy.deepcopy(tf_adata.uns['pls']['pls_mod']) for tc in test_conds}
        umap_models = {tc: copy.deepcopy(tf_adata.uns['umap']['umap_mod']) for tc in test_conds}

        tf_adata_merged = project_pls_per_condition(
            tf_adata = tf_adata_merged,
            pls_models = pls_models, 
            umap_models = None, #umap_models, <-- projecting into new UMAP is problematic
            ctrl_pert = ctrl_pert, # only needed for per_condition_models
            counterfactual = 'perturbation', 
            cat_counterfactual_map = None,
        )

        remove_type_ = remove_type
        if type(remove_type) == list:
            remove_type_ = '^'.join(remove_type)    
        predictions_res[remove_type_ + '_{}'.format(fold)] = tf_adata_merged

    loss_res = pd.DataFrame(loss_res)
    loss_res.remove_type = loss_res.remove_type.apply(lambda x: '^'.join(x) if type(x) == list else x)

    return loss_res, predictions_res

In [194]:
iterables = list(itertools.product(mod_types, folds, remove_types))

# only do remove type not None for the full model
iterables = [it for it in iterables if ((it[0] == 'actual') or it[2] == 'none')]


loss_res_all = []
predictions_res_all = []
for (mod_type, fold, remove_type) in iterables:
    loss_res, predictions_res = prediction_pipeline(fold = fold, mod_type = mod_type, remove_type = remove_type)
    loss_res_all.append(loss_res)

    if mod_type != 'actual':
        assert predictions_res is None, 'Baseline scLEMBAS models should not have merged AnnData objects returned'
    else:
        predictions_res_all.append(predictions_res)

        
loss_res_all = pd.concat(loss_res_all)
loss_res_all.to_csv(os.path.join(data_path, 'processed', '{}_scLEMBAS_model_losses.csv'.format(author)))

predictions_res_all = {k: v for d in predictions_res_all for k, v in d.items()}
io.write_pickled_object(predictions_res_all, 
                       os.path.join(data_path, 'processed', '{}_scLEMBAS_model_predictions.pickle'.format(author)))

# Psuedo-bulk losses

Predictions were actually generated in the training script, so here we just calculate the loss. 

In [172]:
from sklearn.metrics import root_mean_squared_error
from scipy import stats

def pearson_delta_(y_actual, y_pred, groupby_type):
    """Emulates that of `scLEMBAS.metrics.distances.pearson_delta`, but without the initial psuedobulking setp."""
    if groupby_type == 'perturbation':
        assert ctrl_pert not in y_pred.index, 'PearsonDelta needs a control reference'
        y_control = y_actual.loc[ctrl_pert, :]
    elif groupby_type == 'condition':
        assert ctrl_pert not in [cond.split('^')[1] for cond in y_pred.index], 'PearsonDelta needs a control reference'
        y_control = y_actual.loc[[cond for cond in y_actual.index if cond.endswith('^{}'.format(ctrl_pert))], :]
        y_control.index = [cond.split('^')[0] for cond in y_control.index] 
        y_control = y_control.loc[[cond.split('^')[0] for cond in y_pred.index] , :]

    y_actual = y_actual.loc[y_pred.index, :] # filter for conditions present in predicted

    actual_delta = np.subtract(y_actual.values, y_control.values)
    pred_delta = np.subtract(y_pred.values, y_control.values)

    r = np.array([
        stats.pearsonr(a, b)[0]
        for a, b in zip(actual_delta, pred_delta)
    ])
    
    return np.mean(r)

def assert_rows_equal_within_type2(df: pd.DataFrame, sep="^", atol=1e-5, rtol=1e-8, equal_nan=True):
    type2 = df.index.to_series().str.split(sep, n=1).str[1]
    all_ok = True
    for t2, idx in type2.groupby(type2).groups.items():
        sub = df.loc[idx]
        ref = sub.iloc[0].to_numpy()
        arr = sub.to_numpy()
        ok = np.allclose(arr, ref[None, :], atol=atol, rtol=rtol, equal_nan=equal_nan)
        if not ok:
            all_ok = False
            break
    return all_ok


def pb_pipeline(fold, baseline_type):

    split = get_split(fold, author)

    fn_pb = os.path.join(data_path, 'interim', '{}_fold{}_{}baseline_prediction.csv'.format(author, fold, baseline_type))

    y_pred_baseline = pd.read_csv(fn_pb, index_col = 0)
    y_pred_baseline_pert = y_pred_baseline.copy()

    if baseline_type == 'RF': # psueo-bulk post-hoc
        y_pred_baseline_pert[pert_col] = tf_adata.obs.loc[y_pred_baseline.index, pert_col]
        y_pred_baseline_pert = y_pred_baseline_pert.groupby(pert_col, observed = True).mean()

        # will be the same across cell types, but this is for comparison with psuedo-bulk types
        y_pred_baseline_condition = y_pred_baseline.copy()
        y_pred_baseline_condition['condition'] = tf_adata.obs.loc[y_pred_baseline.index, 'condition']
        y_pred_baseline_condition = y_pred_baseline_condition.groupby('condition', observed = True).mean()
    elif baseline_type == 'linear':  # already psueo-bulked
        y_pred_baseline_pert = y_pred_baseline_pert.T

        # is the same across cell types, but this is for comparison with psuedo-bulk types
        y_pred_baseline_condition = pd.DataFrame(index = split['test_conds'], columns = tf_adata.var_names)
        for test_cond in y_pred_baseline_condition.index:
            y_pred_baseline_condition.loc[test_cond, :] = y_pred_baseline_pert.loc[test_cond.split('^')[1], :].values
            y_pred_baseline_condition = y_pred_baseline_condition.astype(np.float64)

    elif baseline_type == 'mean':
        test_perts = sorted({test_cond.split('^')[1] for test_cond in split['test_conds']})
        y_pred_baseline_pert = pd.DataFrame(index = test_perts, columns = tf_adata.var_names)
        for idx in y_pred_baseline_pert.index:
            y_pred_baseline_pert.loc[idx, :] = y_pred_baseline['0'].values
            y_pred_baseline_pert = y_pred_baseline_pert.astype(np.float64)

        y_pred_baseline_condition = pd.DataFrame(index = split['test_conds'], columns = tf_adata.var_names)
        for idx in y_pred_baseline_condition.index:
            y_pred_baseline_condition.loc[idx, :] = y_pred_baseline['0'].values
            y_pred_baseline_condition = y_pred_baseline_condition.astype(np.float64)

    assert assert_rows_equal_within_type2(y_pred_baseline_condition, atol = 1e-5, rtol = 1e-8), 'Condition baselines should be the same across perturbations'


    # psueo-bulked actual data and losses
    y_actual_pert = distances.psuedobulk_adata(adata = tf_adata, groupby_col = pert_col)

    pd_pert = pearson_delta_(y_actual = y_actual_pert, y_pred=y_pred_baseline_pert, groupby_type = 'perturbation')
    rmse_pert = root_mean_squared_error(y_actual_pert.loc[y_pred_baseline_pert.index, :], 
                                        y_pred_baseline_pert, 
                                        multioutput = 'uniform_average')

    y_actual_condition = distances.psuedobulk_adata(adata = tf_adata, groupby_col = 'condition')
    pd_condition = pearson_delta_(y_actual = y_actual_condition, y_pred=y_pred_baseline_condition, groupby_type = 'condition')
    rmse_condition = root_mean_squared_error(y_actual_condition.loc[y_pred_baseline_condition.index, :], 
                                             y_pred_baseline_condition, multioutput = 'uniform_average')
    
    loss_res = pd.DataFrame(
        data = {'loss': [rmse_pert, pd_pert, rmse_condition, pd_condition],
                'loss_type': ['RMSE_{}'.format(pert_col), 'PearsonDelta_{}'.format(pert_col), 
                             'RMSE_condition', 'PearsonDelta_condition'], 
                'remove_type': ['none']*4, 
                'fold': [fold]*4, 
                'model_type': ['psuedobulk_{}_baseline'.format(baseline_type)]*4
               }
    )

    return loss_res



In [175]:
folds = range(5)
baseline_types = ['RF', 'linear','mean']

iterables = itertools.product(baseline_types, folds)

loss_res_pb = []
for (baseline_type, fold) in iterables:
    loss_res = pb_pipeline(fold, baseline_type)
    loss_res_pb.append(loss_res)
    
loss_res_pb = pd.concat(loss_res_pb)
loss_res_pb.to_csv(os.path.join(data_path, 'processed', '{}_psuedobulk_baseline_losses.csv'.format(author)))

In [180]:
loss_res_pb

,loss,loss_type,remove_type,fold,model_type
0,0.075922,RMSE_ligand,none,0,psuedobulk_RF_baseline
1,0.933671,PearsonDelta_ligand,none,0,psuedobulk_RF_baseline
2,0.284932,RMSE_condition,none,0,psuedobulk_RF_baseline
3,0.623551,PearsonDelta_condition,none,0,psuedobulk_RF_baseline
0,0.080608,RMSE_ligand,none,1,psuedobulk_RF_baseline
1,0.861827,PearsonDelta_ligand,none,1,psuedobulk_RF_baseline
2,0.187876,RMSE_condition,none,1,psuedobulk_RF_baseline
3,0.703126,PearsonDelta_condition,none,1,psuedobulk_RF_baseline
0,0.100864,RMSE_ligand,none,2,psuedobulk_RF_baseline
1,0.923609,PearsonDelta_ligand,none,2,psuedobulk_RF_baseline


In [189]:
loss_res_pb.groupby(['loss_type', 'model_type'])['loss'].median()

loss_type               model_type                
PearsonDelta_condition  psuedobulk_RF_baseline        0.703126
                        psuedobulk_linear_baseline    0.521079
                        psuedobulk_mean_baseline      0.475105
PearsonDelta_ligand     psuedobulk_RF_baseline        0.923609
                        psuedobulk_linear_baseline    0.721672
                        psuedobulk_mean_baseline      0.560225
RMSE_condition          psuedobulk_RF_baseline        0.256231
                        psuedobulk_linear_baseline    0.243939
                        psuedobulk_mean_baseline      0.225356
RMSE_ligand             psuedobulk_RF_baseline        0.080608
                        psuedobulk_linear_baseline    0.163058
                        psuedobulk_mean_baseline      0.204505
Name: loss, dtype: float64

In [193]:
pb_scl = loss_res_all[loss_res_all.remove_type == 'none']
pb_scl = pb_scl[pb_scl.loss_type != 'EMD']

pb_scl.groupby(['loss_type', 'model_type'])['loss'].median()

loss_type               model_type
PearsonDelta_condition  actual        0.437809
PearsonDelta_ligand     actual        0.359087
RMSE_condition          actual        0.259182
RMSE_ligand             actual        0.297308
Name: loss, dtype: float64

Make some points about psuedobulk baselines > scLEMBAS:
    
1) scLEMBAS is the only model with an improvement when accounting for cell type (condition vs perturbation)
2) Psuedobulk means that no heterogeneity is captured -- this isn't a completely fair comparison. *TODO: Show loss of something with psuedobulking* (maybe want to show 

3) scLEMBAS has a second utility beyond prediction, which is learning the network topology